In [ ]:
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu124
!pip install --no-deps packaging ninja einops flash-attn trl peft accelerate bitsandbytes
!pip install --upgrade --no-cache-dir git+https://github.com/unslothai/unsloth.git
!pip install -U datasets
!pip install -U unsloth_zoo

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 22.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 15.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Fo

In [ ]:
import torch
import os
import json
import pandas as pd
import trl
from datasets import Dataset, DatasetDict
from datasets import load_dataset
from huggingface_hub import notebook_login
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

<ipython-input-2-7e3015d3ddb8>:11: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Step 3.** Login to Your Hugging Face with hf_token. (write access token)

In [ ]:
notebook_login()

# **Step 4.** Convert your JSON dataset to Llama3 finetuning format


In [ ]:
huggingface_user = "BAC3030"
dataset_name = "hinrik_lp500"

# class Llama3InstructDataset:
#     def __init__(self, data):
#         self.data = data
#         self.prompts = []
#         self.create_prompts()

#     def create_prompt(self, row):
#         prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>{row['instruction']}<|eot_id|><|start_header_id|>user<|end_header_id|>{row['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>{row['output']}<|eot_id|>"""
#         return prompt

#     def create_prompts(self):
#         for row in self.data:
#             prompt = self.create_prompt(row)
#             self.prompts.append(prompt)

#     def get_dataset(self):
#         df = pd.DataFrame({'prompt': self.prompts})
#         return df

# def create_dataset_hf(dataset):
#     dataset.reset_index(drop=True, inplace=True)
#     return DatasetDict({"train": Dataset.from_pandas(dataset)})

# if __name__ == "__main__":
#     with open('/content/dataset.json', 'r', encoding='utf-8') as f:
#         data = json.load(f)

#     dataset = Llama3InstructDataset(data)
#     df = dataset.get_dataset()

#     processed_data_path = 'processed_data'
#     os.makedirs(processed_data_path, exist_ok=True)

#     llama3_dataset = create_dataset_hf(df)
#     llama3_dataset.save_to_disk(os.path.join(processed_data_path, "llama3_dataset"))
#     llama3_dataset.push_to_hub(f"{huggingface_user}/{dataset_name}")

Saving the dataset (0/1 shards):   0%|          | 0/502 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

No files have been modified since last commit. Skipping to prevent empty commit.


# **Step 5.** LoRa Finetuning Configurations
- "finetuned_model" sets your models name on HF
- "num_train_epochs" sets the number of epochs for training

    (epoch = 1 pass through your entire dataset)

In [ ]:
# Defining the configuration for the base model, LoRA and training
config = {
    "hugging_face_username":huggingface_user,
    "model_config": {
        "base_model":"mistralai/Mistral-7B-Instruct-v0.3", # The base model
        "finetuned_model":"hinrik-mistral-kaggle-100steps", # The finetuned model
        "max_seq_length": 2048, # The maximum sequence length
        "dtype":torch.float16, # The data type
        "load_in_4bit": True, # Load the model in 4-bit
    },
    "lora_config": {
      "r": 16, # The number of LoRA layers 8, 16, 32, 64
      "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # The target modules
      "lora_alpha":16, # The alpha value for LoRA
      "lora_dropout":0, # The dropout value for LoRA
      "bias":"none", # The bias for LoRA
      "use_gradient_checkpointing":True, # Use gradient checkpointing
      "use_rslora":False, # Use RSLora
      "use_dora":False, # Use DoRa
      "loftq_config":None # The LoFTQ configuration
    },
    "training_dataset":{
        "name":f"{huggingface_user}/{dataset_name}", # The dataset name(huggingface/datasets)
        "split":"train", # The dataset split
        "input_field":"prompt", # The input field
    },
    "training_config": {
      "per_device_train_batch_size": 8, # The batch size
      "gradient_accumulation_steps": 4, # The gradient accumulation steps
      "warmup_steps": 10, # The warmup steps
      "max_steps": 100, # The maximum steps (0 if the epochs are defined)
      "num_train_epochs": 50, # The number of training epochs(0 if the maximum steps are defined)
      "learning_rate": 2e-4, # The learning rate
      "fp16": not torch.cuda.is_bf16_supported(),  # The fp16
      "bf16": torch.cuda.is_bf16_supported(), # The bf16
      "logging_steps": 1, # The logging steps
      "optim" :"adamw_8bit", # The optimizer
      "weight_decay" : 0.01,  # The weight decay
      "lr_scheduler_type": "linear", # The learning rate scheduler
      "seed" : 42, # The seed
      "output_dir" : "outputs" # The output directory
    }
}

# **Step 6.** Load Llama3-8B, QLoRA & Trainer Model

In [ ]:
# Loading the model and the tokinizer for the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config.get("model_config").get("base_model"),
    max_seq_length = config.get("model_config").get("max_seq_length"),
    dtype = config.get("model_config").get("dtype"),
    load_in_4bit = config.get("model_config").get("load_in_4bit"),
)

# Setup for QLoRA/LoRA peft of the base model
model = FastLanguageModel.get_peft_model(
    model,
    r = config.get("lora_config").get("r"),
    target_modules = config.get("lora_config").get("target_modules"),
    lora_alpha = config.get("lora_config").get("lora_alpha"),
    lora_dropout = config.get("lora_config").get("lora_dropout"),
    bias = config.get("lora_config").get("bias"),
    use_gradient_checkpointing = config.get("lora_config").get("use_gradient_checkpointing"),
    random_state = 42,
    use_rslora = config.get("lora_config").get("use_rslora"),
    use_dora = config.get("lora_config").get("use_dora"),
    loftq_config = config.get("lora_config").get("loftq_config"),
)

# Loading the training dataset
# dataset_train = load_dataset(config.get("training_dataset").get("name"), split = config.get("training_dataset").get("split"))
dataset_train = load_dataset("json", data_files="dataset.json", split="train")

from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False)

# Setting up the trainer for the model
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset_train,
    dataset_num_proc = 2,
    packing = False,
    data_collator = data_collator,
    args = SFTConfig(
        dataset_text_field = config.get("training_dataset").get("input_field"),
        max_seq_length = config.get("model_config").get("max_seq_length"),
        per_device_train_batch_size = config.get("training_config").get("per_device_train_batch_size"),
        gradient_accumulation_steps = config.get("training_config").get("gradient_accumulation_steps"),
        warmup_steps = config.get("training_config").get("warmup_steps"),
        max_steps = config.get("training_config").get("max_steps"),
        num_train_epochs= config.get("training_config").get("num_train_epochs"),
        learning_rate = config.get("training_config").get("learning_rate"),
        fp16 = config.get("training_config").get("fp16"),
        bf16 = config.get("training_config").get("bf16"),
        logging_steps = config.get("training_config").get("logging_steps"),
        optim = config.get("training_config").get("optim"),
        weight_decay = config.get("training_config").get("weight_decay"),
        lr_scheduler_type = config.get("training_config").get("lr_scheduler_type"),
        seed = 42,
        output_dir = config.get("training_config").get("output_dir"),
        report_to = "none",
    ),
)

==((====))==  Unsloth 2025.4.8: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth: Tokenizing ["prompt"] (num_proc=2):   0%|          | 0/502 [00:00<?, ? examples/s]

# **Step 7.** Train Your Finetuned Model

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 502 | Num Epochs = 10 | Total steps = 150
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss
1,4.850300
2,4.903600
3,4.795600
4,4.488900
5,4.015900
6,3.649900
7,3.128000
8,2.630800
9,2.343300
10,1.900400


# **Step 8.** Save Trainer Stats

In [ ]:
with open("trainer_stats.json", "w") as f:
    json.dump(trainer_stats, f, indent=4)

# **Step 9.** Save Finetuned Model & Push to HF Hub

In [ ]:
model.push_to_hub_gguf("hinrik-lp500-150-steps", tokenizer, quantization_method = "q4_k_m")

Unsloth: Will remove a cached repo with size 6.0G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 3.14 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [03:24<00:00,  6.41s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving hinrik-lp500-150-steps/pytorch_model-00001-of-00004.bin...
Unsloth: Saving hinrik-lp500-150-steps/pytorch_model-00002-of-00004.bin...
Unsloth: Saving hinrik-lp500-150-steps/pytorch_model-00003-of-00004.bin...
Unsloth: Saving hinrik-lp500-150-steps/pytorch_model-00004-of-00004.bin...
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at hinrik-lp500-150-steps into f16 GGUF format.
The output location will be /content/hinrik-lp500-150-steps/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: hinrik-lp500-150-steps
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.json'
INFO:hf-to-gguf:gguf: loading model

unsloth.Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Saved GGUF to https://huggingface.co/BAC3030/hinrik-lp500-150-steps


# **Step 10.** Test your pretrained model in Colab

In [ ]:
# # Loading the fine-tuned model and the tokenizer for inference
# model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name = config.get("model_config").get("finetuned_model"),
#         max_seq_length = config.get("model_config").get("max_seq_length"),
#         dtype = config.get("model_config").get("dtype"),
#         load_in_4bit = config.get("model_config").get("load_in_4bit"),
#     )

# # Using FastLanguageModel for fast inference
# FastLanguageModel.for_inference(model)

# system_prompt = f"You are an AI that translates english sentences to ASL gloss. Always assume that the input is an english sentence that is to be translated to english. You will translate the inputted sentence into ASL gloss."

# # Tokenizing the input and generating the output
# prompt = input('I declare the session of the european parliament adjourned.')
# inputs = tokenizer(
# [
#     f"<|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>{prompt}<|end_header_id|>"
# ], return_tensors = "pt").to("cuda")
# outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
# tokenizer.batch_decode(outputs, skip_special_tokens = True)